# 🕸️ 00 · 图基础与图表示学习：网络X + 随机游走 + Node2Vec

> 图神经网络（GNN）从**图的表示**开始：把节点/边/子图变成向量。
> 本节用 networkx 建图、可视化、算度与连通分量，再手写**随机游走（DeepWalk 思想）**
> 与 **Node2Vec 的 BFS/DFS 游走**——这是图表示学习的起点。

## 📋 本节要解决的问题

| # | 问题 | 方法 |
|---|---|---|
| 1 | 图怎么表示？ | 邻接矩阵 / 邻接表 |
| 2 | 图的基本性质？ | 度/路径/连通分量/聚类系数 |
| 3 | 节点怎么变向量？ | 随机游走 + word2vec 思想 |
| 4 | BFS vs DFS 游走？ | Node2Vec 两个超参 p/q |

In [1]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Noto Sans CJK SC', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np
import pandas as pd
import networkx as nx
import sqlite3
import os
rng = np.random.default_rng(42)


## 1️⃣ 建图与可视化（networkx）

In [2]:
G = nx.Graph()
edges = [('A', 'B'), ('A', 'C'), ('B', 'D'), ('C', 'D'), ('C', 'E'),
         ('D', 'E'), ('E', 'F'), ('F', 'G'), ('G', 'H'), ('F', 'H')]
G.add_edges_from(edges)
print('节点数:', G.number_of_nodes(), '| 边数:', G.number_of_edges())
print('节点度:', dict(G.degree()))
print('平均聚类系数:', round(nx.average_clustering(G), 3))

plt.figure(figsize=(6, 5))
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, with_labels=True, node_color='#2563eb', node_size=600,
        font_color='white', font_size=12)
plt.title('社交/引用关系图（示例）')
plt.show()
assert G.number_of_edges() == 10
print('✅ 图构建与可视化完成')

节点数: 8 | 边数: 10
节点度: {'A': 2, 'B': 2, 'C': 3, 'D': 3, 'E': 3, 'F': 3, 'G': 2, 'H': 2}
平均聚类系数: 0.417
✅ 图构建与可视化完成


C:\Users\24260\AppData\Local\Temp\ipykernel_47624\1107041722.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2️⃣ 图的矩阵表示：邻接矩阵

In [3]:
nodes = sorted(G.nodes())
A = nx.to_numpy_array(G, nodelist=nodes)
print('邻接矩阵 (8x8):')
print(A.astype(int))
print('\n行和（度）:', A.sum(axis=1).astype(int))
# 加自环：GNN 里常用（让节点聚合时包含自己）
A_hat = A + np.eye(len(nodes))
print('加自环后的对角:', np.diag(A_hat).astype(int))
assert A.shape == (8, 8) and np.allclose(A, A.T)
print('✅ 邻接矩阵对称（无向图）')

邻接矩阵 (8x8):
[[0 1 1 0 0 0 0 0]
 [1 0 0 1 0 0 0 0]
 [1 0 0 1 1 0 0 0]
 [0 1 1 0 1 0 0 0]
 [0 0 1 1 0 1 0 0]
 [0 0 0 0 1 0 1 1]
 [0 0 0 0 0 1 0 1]
 [0 0 0 0 0 1 1 0]]

行和（度）: [2 2 3 3 3 3 2 2]
加自环后的对角: [1 1 1 1 1 1 1 1]
✅ 邻接矩阵对称（无向图）


## 3️⃣ 图论基本性质（面试速查）

In [4]:
print('''核心概念速答：
  度：节点的边数（无向）；入度/出度（有向）；
  路径/最短路：BFS 求无权最短路；
  连通分量：互相可达的节点集合；
  聚类系数：我的邻居之间也是邻居的比例（小世界性质）；
  桥（割边）：删除后图不连通 -> 网络关键边；
  中心性：度中心/介数中心（信息流通枢纽）。
''')
shortest = nx.shortest_path(G, 'A', 'H')
print('A->H 最短路:', shortest, '长度:', len(shortest) - 1)
print('连通分量:', [c for c in nx.connected_components(G)])
bridges = list(nx.bridges(G))
print('割边（桥）:', bridges, '数量:', len(bridges))

核心概念速答：
  度：节点的边数（无向）；入度/出度（有向）；
  路径/最短路：BFS 求无权最短路；
  连通分量：互相可达的节点集合；
  聚类系数：我的邻居之间也是邻居的比例（小世界性质）；
  桥（割边）：删除后图不连通 -> 网络关键边；
  中心性：度中心/介数中心（信息流通枢纽）。

A->H 最短路: ['A', 'C', 'E', 'F', 'H'] 长度: 4
连通分量: [{'C', 'G', 'D', 'A', 'E', 'F', 'H', 'B'}]
割边（桥）: [('E', 'F')] 数量: 1


## 4️⃣ 随机游走生成"句子"（DeepWalk 思想）

In [5]:
# 图节点是字符 A..H，先映射为索引再游走
node_list = list(G.nodes())

def random_walk(G, node_list, start_idx, length=6, seed=42):
    r = np.random.default_rng(seed)
    walk = [start_idx]
    cur = start_idx
    for _ in range(length - 1):
        nbrs = [node_list.index(nb) for nb in G.neighbors(node_list[cur])]
        if not nbrs:
            break
        cur = int(r.choice(nbrs))
        walk.append(cur)
    return walk

walks = []
for _ in range(20):
    s = int(rng.integers(0, len(node_list)))
    walks.append(random_walk(G, node_list, s, length=6))
print('示例游走:', [chr(65 + w) for w in walks[0]])
print('全部游走序列数:', len(walks), '总步数:', sum(len(w) for w in walks))
print('➡️ 游走序列 ≈ 自然语言"句子"，节点≈词：再用 word2vec 学 embedding')

示例游走: ['A', 'B', 'D', 'C', 'D', 'C']
全部游走序列数: 20 总步数: 120
➡️ 游走序列 ≈ 自然语言"句子"，节点≈词：再用 word2vec 学 embedding


## 5️⃣ Node2Vec：BFS（同质）与 DFS（结构）游走

In [6]:
print('''Node2Vec 两个超参：
  p（Return）：控制回到上一节点的概率（p 大 = 不走回头路，倾向 DFS）；
  q（In-out）：控制走向邻居还是更远（q 大 = BFS 倾向，q 小 = DFS 倾向）；
  BFS 游走 -> 学习"角色/结构"（如两个不同社区的门卫相似）；
  DFS 游走 -> 学习"同质/邻近"（相连的节点向量相近）。
''')
# 简化演示：从 E 出发，看不同 p/q 下的 2 步选择
def n2v_step(G, node_list, prev, cur, p=1.0, q=1.0, seed=42):
    r = np.random.default_rng(seed)
    nbrs = list(G.neighbors(cur))
    weights = []
    for nxt in nbrs:
        if nxt == prev:
            w = 1.0 / p      # 返回
        elif G.has_edge(prev, nxt) or G.has_edge(nxt, prev):
            w = 1.0          # 与上一节点相连
        else:
            w = 1.0 / q      # 其他
        weights.append(w)
    chosen = r.choice(nbrs, p=np.array(weights) / sum(weights))
    return node_list.index(str(chosen))

print('p=0.5(爱返回) E 的下一步:', chr(65 + n2v_step(G, node_list, 'D', 'E', p=0.5, q=1.0, seed=1)))
print('p=2.0(避返回) E 的下一步:', chr(65 + n2v_step(G, node_list, 'D', 'E', p=2.0, q=1.0, seed=1)))
print('✅ Node2Vec 通过 p/q 控制游走语义')

Node2Vec 两个超参：
  p（Return）：控制回到上一节点的概率（p 大 = 不走回头路，倾向 DFS）；
  q（In-out）：控制走向邻居还是更远（q 大 = BFS 倾向，q 小 = DFS 倾向）；
  BFS 游走 -> 学习"角色/结构"（如两个不同社区的门卫相似）；
  DFS 游走 -> 学习"同质/邻近"（相连的节点向量相近）。

p=0.5(爱返回) E 的下一步: D
p=2.0(避返回) E 的下一步: D
✅ Node2Vec 通过 p/q 控制游走语义


## 6️⃣ 图表示 vs 词向量（面试对比）

In [7]:
print('''| 维度 | 词向量 (word2vec) | 图表示 (DeepWalk/Node2Vec) |
|---|---|---|
| 语料 | 句子序列 | 随机游走序列 |
| 上下文 | 窗口内词 | 窗口内节点 |
| 模型 | skip-gram/CBOW | 同（套用） |
| 目标 | 词共现 | 节点共现/结构 |
| 局限 | 词序假设 | 游走策略依赖 |

后续 GNN 用"消息传递"替代游走+word2vec，可直接上监督任务。
''')

| 维度 | 词向量 (word2vec) | 图表示 (DeepWalk/Node2Vec) |
|---|---|---|
| 语料 | 句子序列 | 随机游走序列 |
| 上下文 | 窗口内词 | 窗口内节点 |
| 模型 | skip-gram/CBOW | 同（套用） |
| 目标 | 词共现 | 节点共现/结构 |
| 局限 | 词序假设 | 游走策略依赖 |

后续 GNN 用"消息传递"替代游走+word2vec，可直接上监督任务。



## 📝 自测清单

- [ ] 能用 networkx 建图并计算度/最短路/连通分量/聚类系数
- [ ] 能写随机游走并解释 DeepWalk 思想
- [ ] 能解释 Node2Vec 的 p/q 与 BFS/DFS 语义
- [ ] 能对比图表示与词向量